# Mass budget

A worked example of the `quicksat` mass budget against the sample satellite in this
directory: a small Earth observation platform with a telescope payload, hydrazine
propulsion, and a separation interface split between the satellite and the launcher.

In [1]:
from pathlib import Path

import pandas as pd

from quicksat.mass.budget import MassBudget, MassCase

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

# works whether the notebook is run from the repo root or from sample/
SAMPLE = Path.cwd()
if not (SAMPLE / "equipment.csv").exists():
    SAMPLE = SAMPLE / "sample"

budget = MassBudget.from_csv(SAMPLE / "equipment.csv", SAMPLE / "budget_config.yaml")
f"{len(budget.equipment)} rows, harness included"

'30 rows, harness included'

## The equipment table

The CSV is flat. `location` is the computation axis — it drives the system margin, the
harness fraction, and what survives separation. `responsibility` and `subsystem` are
there purely so the budget can be summed different ways.

Masses were parsed with pint on load, so `750 g` for the IMU has already become
kilograms, and the harness rows at the bottom were derived rather than typed in.

In [2]:
budget.equipment[
    [
        "equipment_id",
        "location",
        "responsibility",
        "subsystem",
        "unit_mass_kg",
        "number_of_units",
        "equipment_margin",
        "mass_class",
        "cbe_kg",
    ]
]

,equipment_id,location,responsibility,subsystem,unit_mass_kg,number_of_units,equipment_margin,mass_class,cbe_kg
0,star_tracker,Platform,Platform,ADCS,1.20,2,5.00,equipment,2.40
1,reaction_wheel,Platform,Platform,ADCS,7.00,4,10.00,equipment,28.00
2,magnetorquer,Platform,Platform,ADCS,1.10,3,10.00,equipment,3.30
3,imu,Platform,Platform,ADCS,0.75,1,5.00,equipment,0.75
4,gps_receiver,Platform,Platform,ADCS,1.60,1,10.00,equipment,1.60
5,battery,Platform,Platform,EPS,12.50,1,15.00,equipment,12.50
6,solar_array,Platform,Platform,EPS,9.40,2,20.00,equipment,18.80
7,pcdu,Platform,Platform,EPS,14.00,1,10.00,equipment,14.00
8,obc,Platform,Platform,OBDH,3.20,2,5.00,equipment,6.40
9,mass_memory,Platform,Platform,OBDH,4.50,1,15.00,equipment,4.50


## The four mass cases

Two switches — where we are in the mission, and whether propellant is loaded — give
the four masses usually quoted for a satellite.

In [3]:
cases = {
    "launch mass (on ground, wet)": (MassCase.ON_GROUND, True),
    "dry mass at launch": (MassCase.ON_GROUND, False),
    "separated wet mass (in orbit)": (MassCase.IN_ORBIT, True),
    "in-orbit dry mass": (MassCase.IN_ORBIT, False),
}

pd.DataFrame(
    [
        {
            "case": label,
            "mass [kg]": budget.total(case=case, with_propellant=wet).to("kg").magnitude,
        }
        for label, (case, wet) in cases.items()
    ]
).set_index("case")

,mass [kg]
case,
"launch mass (on ground, wet)",457.35
dry mass at launch,435.35
separated wet mass (in orbit),440.63
in-orbit dry mass,418.63


The difference between the first and third rows is exactly the launcher-side hardware:
the adapter ring half and the clampband that stay behind at separation.

In [4]:
on_ground = budget.total(case=MassCase.ON_GROUND).to("kg")
in_orbit = budget.total(case=MassCase.IN_ORBIT).to("kg")

left_behind = budget.equipment.query("location == 'Launcher'")
print(f"on ground   {on_ground:~.2f}")
print(f"in orbit    {in_orbit:~.2f}")
print(f"difference  {(on_ground - in_orbit):~.2f}")
print()
print(left_behind[["equipment_id", "equipment_name", "cbe_kg"]].to_string(index=False))

on ground   457.35 kg
in orbit    440.63 kg
difference  16.72 kg

equipment_id              equipment_name  cbe_kg
 sep_ring_lv Adapter ring, launcher side   11.00
   clampband         Clampband and pyros    4.20


## Summing over the three axes

Each view is the same set of rows grouped a different way, so all three reconcile to
the same total. `cbe` is the raw estimate, `mev` adds the per-item margin, and `total`
adds the location's system margin on top.

In [5]:
budget.by_subsystem(case=MassCase.IN_ORBIT, with_propellant=True)

,cbe,mev,total
subsystem,,,
ADCS,36.05,39.50,47.40
COMM,12.70,14.06,16.88
EPS,45.30,52.34,62.80
Harness,12.89,14.18,16.87
OBDH,10.90,11.89,14.27
Payload,81.50,96.42,110.89
Propulsion,32.20,33.66,35.99
Structure,83.00,98.70,118.44
Thermal,12.10,14.41,17.09


In [6]:
budget.by_location(case=MassCase.IN_ORBIT, with_propellant=True)

,cbe,mev,total
location,,,
Payload,87.45,103.31,118.80
Platform,239.19,271.86,321.83


In [7]:
budget.by_responsibility(case=MassCase.IN_ORBIT, with_propellant=True)

,cbe,mev,total
responsibility,,,
Harness,12.89,14.18,16.87
Payload,84.90,100.50,115.58
Platform,228.85,260.48,308.18


Worth checking rather than assuming — the three groupings must agree in every case:

In [8]:
for label, (case, wet) in cases.items():
    kwargs = {"case": case, "with_propellant": wet}
    total = budget.total(**kwargs).to("kg").magnitude
    views = [
        budget.by_location(**kwargs)["total"].sum(),
        budget.by_responsibility(**kwargs)["total"].sum(),
        budget.by_subsystem(**kwargs)["total"].sum(),
    ]
    agree = all(abs(view - total) < 1e-9 for view in views)
    print(f"{label:32s} {total:8.2f} kg   three views agree: {agree}")

launch mass (on ground, wet)       457.35 kg   three views agree: True
dry mass at launch                 435.35 kg   three views agree: True
separated wet mass (in orbit)      440.63 kg   three views agree: True
in-orbit dry mass                  418.63 kg   three views agree: True


## Harness

Harness is never typed into the CSV. One row per location is derived as a percentage
of that location's *equipment* CBE — propellant is excluded, since cabling scales with
the boxes it connects — and then carries its own contingency.

Because the harness row is keyed on location just like everything else, it picks up
that location's system margin without any special handling, and shows up as its own
line in the subsystem view above.

In [9]:
budget.equipment.query("subsystem == 'Harness'")[
    ["equipment_id", "location", "cbe_kg", "equipment_margin", "comments"]
]

,equipment_id,location,cbe_kg,equipment_margin,comments
28,harness_payload,Payload,2.55,10.00,Derived: 3.0% of 84.900 kg equipment CBE
29,harness_platform,Platform,10.34,10.00,Derived: 5.0% of 206.850 kg equipment CBE


## Where the margins land

Splitting the in-orbit dry mass into its three layers shows how much of the budget is
estimate and how much is contingency.

In [10]:
frame = budget.resolve(case=MassCase.IN_ORBIT, with_propellant=False)

layers = pd.DataFrame(
    {
        "mass [kg]": [
            frame["cbe_kg"].sum(),
            frame["mev_kg"].sum() - frame["cbe_kg"].sum(),
            frame["total_kg"].sum() - frame["mev_kg"].sum(),
        ]
    },
    index=["current best estimate", "equipment margin", "system margin"],
)
layers.loc["in-orbit dry mass"] = layers["mass [kg]"].sum()
layers

,mass [kg]
current best estimate,304.64
equipment margin,48.53
system margin,65.47
in-orbit dry mass,418.63
